In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error




file_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(file_path)




In [ ]:
# Task 2: Write your code here:
print(df.head())

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
print(df.describe())

In [ ]:
# Task 5: Write your code here:
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.histplot(df['Delivery_Time'], kde=True, color='skyblue')
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
#cheking missing values
missing_info = df.isnull().sum()
print(missing_info[missing_info > 0])

#handle
df = df.fillna(df.median(numeric_only=True))
df = df.apply(lambda x: x.fillna(x.value_counts().index[0]) if x.dtype == "object" else x)


In [ ]:
# Task 3: Write your code here:
#checking
duplicates_count = df.duplicated().sum()
print(f"Number of duplicates: {duplicates_count}")

#removing duplicates
df = df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
#identifying
cat_cols = df.select_dtypes(include=['object', 'category']).columns

#one hot encoding
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

#scaling all features
features = df.columns
df[features] = scaler.fit_transform(df[features])

In [ ]:
# Task 6: Write your code here:
#replace
target_counts = df['Delivery_Time'].value_counts(normalize=True) * 100
print(target_counts)

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np

#split the dataset into features (X) and target (y)
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']



In [ ]:
# Task 2,3,4,5: Write your code here:
#initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

mae_scores = cross_val_score(
    rf_model, X, y,
    cv=kf,
    scoring='neg_mean_absolute_error'
)

average_mae = np.mean(np.abs(mae_scores))
print(f"averaged score across all folds: {average_mae:.4f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

sns.set_theme(style="whitegrid")

def plot_importance(model, X):
    importances = model.feature_importances_
    feature_names = X.columns

    # Organize data
    df_importance = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    df_importance = df_importance.sort_values(by='Importance', ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Feature', data=df_importance, hue='Feature', palette='magma', legend=False)
    plt.title('Top Features Driving Delivery Time Predictions')
    plt.xlabel('Importance Score')
    plt.show()

plot_importance(rf_model, X)

In [ ]:
# Task 2: Write your code here:
def plot_delivery_distribution(y_pred):
    plt.figure(figsize=(10, 6))

    #histogram
    sns.histplot(y_pred, kde=True, color='teal', bins=30, alpha=0.7)

    plt.axvline(np.mean(y_pred), color='red', linestyle='--', label=f'Mean: {np.mean(y_pred):.2f}')

    plt.title('Frequency Distribution of Predicted Delivery Times')
    plt.xlabel('Predicted Time (Minutes)')
    plt.ylabel('Number of Deliveries')
    plt.legend()
    plt.show()



plot_delivery_distribution(y)


In [ ]:
# Task Bonus: Write your code here:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

clear_output()
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
cb_model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, verbose=0, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    rf_model.fit(X_train, y_train)
    cb_model.fit(X_train, y_train)

    rf_preds = rf_model.predict(X_test)
    cb_preds = cb_model.predict(X_test)

    ensemble_preds = (rf_preds + cb_preds) / 2

    fold_mae = mean_absolute_error(y_test, ensemble_preds)
    ensemble_mae_scores.append(fold_mae)

avg_ensemble_mae = np.mean(ensemble_mae_scores)
print(f"Average Ensemble MAE (RF + CatBoost): {avg_ensemble_mae:.4f}")